# Environement

In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML

from eci.decision import response_function
from eci.environment import EnvConfig, Environment

# Local project imports
from eci.plots import (
    animate_belief_trajectory,
    plot_belief_trajectory,
    plot_belief_vote_evolution,
)
from eci.utils import (
    get_voter_trajectory_data,
)
from eci.voting import _vote_plurality, _vote_quadratic

# How observation work?

When you specify a scenario, the function generate observations return a numpy.ndarray (a multi-dimensional array) containing floating-point numbers. The shape of the array is: $(n\_steps, n\_preference)$.

Axis 0 (Rows): Represents time ($n\_steps$).
Axis 1 (Columns): Represents each specific preference ($n\_nodes$).

Values are strictly between $0.0$ and $1.0$. The data represents synthetic observations (probabilities or rates) following a Beta distribution, potentially affected by shocks or trends and Gaussian noise.

If you call generate_observations(n_nodes=3, n_steps=5), the resulting matrix looks like this:

$$\begin{bmatrix}
t_0n_0 & t_0n_1 & t_0n_2 \\
t_1n_0 & t_1n_1 & t_1n_2 \\
t_2n_0 & t_2n_1 & t_2n_2 \\
t_3n_0 & t_3n_1 & t_3n_2 \\
t_4n_0 & t_4n_1 & t_4n_2
\end{bmatrix}$$

In [ ]:
config = EnvConfig(
    num_voters=50, num_candidates=5, num_preferences=1, scenario=2, num_steps=100
)
NUM_SIMULATIONS = 100

In [ ]:
# Initialize environment
env = Environment(config)
env.num_simulations = NUM_SIMULATIONS
env._run_multi_agent_inference()

In [ ]:
env.input_data  # (100 timestep, 2 preference)

In [ ]:
traj_data = get_voter_trajectory_data(env, voter_id=0)
fig, ax1, ax2 = plot_belief_trajectory(**traj_data)

In [ ]:
traj_data = get_voter_trajectory_data(env, voter_id=0)
anim = animate_belief_trajectory(**traj_data, interval=60)
HTML(anim.to_jshtml())

# How to create observations manually?

To replicate the data structure expected by your pipeline without using the generator, you can use NumPy. Here are three ways to do it:

In [ ]:
# Example: 4 time steps, 1 nodes
manual_obs = jnp.arange(start=0.0, stop=1.0, step=0.005).reshape(
    200, 1
)  # (n_steps, n_nodes)

print(f"Shape: {manual_obs.shape}")
# Output: (200, 1) -> (n_steps, n_nodes)

In [ ]:
manual_obs

In [ ]:
# Initialize environment
env = Environment(config)
# Override the input data with our manual observations
env.input_data = manual_obs
# Run the inference with the new observations
env.num_simulations = NUM_SIMULATIONS
env._run_multi_agent_inference()

In [ ]:
traj_data = get_voter_trajectory_data(env, voter_id=0)
anim = animate_belief_trajectory(**traj_data, interval=45)
HTML(anim.to_jshtml())

In [ ]:
traj_data = get_voter_trajectory_data(env, voter_id=0)
fig, ax1, ax2 = plot_belief_trajectory(**traj_data)

# Volatility

ECI has two distinct volatilities in the model:

1. **How Agent manage volatility** (`tonic_volatility_mean`). Each voter's *prior* on how variable the world is. Internally, this is propagated to the **X2 volatility parent** of every HGF state node — not to the state node itself. A common pitfall is to write `tonic_volatility` on `network.input_idxs` (X1), where it is silently ignored.
2. **World volatility** (`obs_dispersion`, `obs_shock_pattern`). The actual turbulence of the observation stream the agents are exposed to.

When prior matches reality, posteriors converge sharply. When they mismatch, agents are miscalibrated — and through the precision-weighted softmax, this changes how decisively they vote.


Two agents see the same noisy world; only their `tonic_volatility_mean` differs. Low-tvm = "I expect a stable world", high-tvm = "I expect rapid change".

In [ ]:
env_low = Environment(
    EnvConfig(
        num_voters=1,
        num_candidates=2,
        num_preferences=1,
        num_steps=100,
        scenario=1,
        tonic_volatility_mean=14,
        seed=42,
        obs_seed=42,
    )
)
env_high = Environment(
    EnvConfig(
        num_voters=1,
        num_candidates=2,
        num_preferences=1,
        num_steps=100,
        scenario=2,
        tonic_volatility_mean=14.0,
        seed=42,
        obs_seed=42,
    )
)
for e in (env_low, env_high):
    e._run_multi_agent_inference()

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, env, title in [
    (axes[0], env_low, "low tonic_volatility (-26)"),
    (axes[1], env_high, "high tonic_volatility (+14)"),
]:
    traj = get_voter_trajectory_data(env, voter_id=0)
    t = np.arange(len(traj["observations"]))
    mean = np.asarray(traj["expected_mean"])
    ci = 1.96 / np.sqrt(np.maximum(np.asarray(traj["expected_precision"]), 1e-9))
    ax.scatter(t, traj["observations"], s=8, c="gray", alpha=0.3, label="observations")
    ax.fill_between(
        t, mean - ci, mean + ci, color="#d62728", alpha=0.15, label="95% CI"
    )
    ax.plot(t, mean, color="#d62728", lw=2, label="belief mean")
    ax.set_title(title)
    ax.set_xlabel("time step")
    ax.grid(True, ls=":", alpha=0.5)
    ax.legend(loc="lower right", fontsize=8)
axes[0].set_ylabel("belief / observation")
plt.tight_layout()
plt.show()

In [ ]:
env_low = Environment(
    EnvConfig(
        num_voters=1,
        num_candidates=2,
        num_preferences=1,
        num_steps=100,
        scenario=2,
        tonic_volatility_mean=-14,
        seed=42,
        obs_seed=42,
    )
)
env_high = Environment(
    EnvConfig(
        num_voters=1,
        num_candidates=2,
        num_preferences=1,
        num_steps=100,
        scenario=2,
        tonic_volatility_mean=14.0,
        seed=42,
        obs_seed=42,
    )
)
for e in (env_low, env_high):
    e._run_multi_agent_inference()

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, env, title in [
    (axes[0], env_low, "low tonic_volatility (-26)"),
    (axes[1], env_high, "high tonic_volatility (+14)"),
]:
    traj = get_voter_trajectory_data(env, voter_id=0)
    t = np.arange(len(traj["observations"]))
    mean = np.asarray(traj["expected_mean"])
    ci = 1.96 / np.sqrt(np.maximum(np.asarray(traj["expected_precision"]), 1e-9))
    ax.scatter(t, traj["observations"], s=8, c="gray", alpha=0.3, label="observations")
    ax.fill_between(
        t, mean - ci, mean + ci, color="#d62728", alpha=0.15, label="95% CI"
    )
    ax.plot(t, mean, color="#d62728", lw=2, label="belief mean")
    ax.set_title(title)
    ax.set_xlabel("time step")
    ax.grid(True, ls=":", alpha=0.5)
    ax.legend(loc="lower right", fontsize=8)
axes[0].set_ylabel("belief / observation")
plt.tight_layout()
plt.show()

# Votes over time: the population's choice as beliefs evolve

So far we have watched a single voter. Here we let the **whole population** vote
at **every timestep** — each voter using its belief *at that step*, with
preferences and candidate policies held fixed — and average the per-candidate
vote share over many stochastic elections.

The new mechanism is `env.vote_share_over_time(response_fn, voting_fn, n_simulations=...)`:
at each timestep it rebuilds the per-agent data from the belief trajectories,
runs the voting rule over `n_simulations` runs, and returns the mean vote share
per candidate, shape `(n_candidates, n_steps)`. This is the population
generalisation of the single-voter vote heatmap (tutorial 5): we can see how a
world shock reshapes the collective vote, and how plurality and quadratic
voting transmit that shift differently.

In [ ]:
# A population on a shocked world that RECOVERS, with observations on the
# candidate scale.
vt_config = EnvConfig(
    num_voters=1,
    num_candidates=5,
    num_preferences=5,
    num_steps=120,
    scenario=2,
    shock_pattern="sudden",
    recover=True,  # ← temporary shock
    dispersion=0.6,
    obs_low=-5.0,
    obs_high=5.0,
    obs_seed=0,
    tonic_volatility_mean=12.0,
    seed=1,
)
vt_env = Environment(vt_config)
vt_env._run_multi_agent_inference()

# Evenly-spaced candidates so the heatmap rows are spatially ordered
# (safe after inference: candidate policies only enter the vote read-out).
even = jnp.linspace(-4.0, 4.0, vt_config.num_candidates).reshape(-1, 1)
for i, c in enumerate(vt_env.candidates):
    c.policy = {"mean": even[i], "precision": c.policy["precision"]}

# P(win) per candidate at each timestep, averaged over stochastic elections.
n_cand = vt_config.num_candidates
plur_win = np.asarray(
    vt_env.vote_outcome_over_time(
        response_function, _vote_plurality, n_simulations=80, metric="win"
    )
)
qv_win = np.asarray(
    vt_env.vote_outcome_over_time(
        response_function,
        _vote_quadratic,
        n_simulations=80,
        metric="win",
        num_votes=None,
    )
)
print("plurality P(win):", plur_win.shape, " QV P(win):", qv_win.shape)

In [ ]:
# Top panel: one representative voter's belief trajectory (same style as
# tutorial 5). The two heatmaps below are the *population* P(win) per
# candidate — how often each candidate wins the election at that timestep.
traj = get_voter_trajectory_data(vt_env, voter_id=0, pref_idx=0)
shock_t = vt_config.num_steps // 3
vmax = float(max(plur_win.max(), qv_win.max()))

plt.rcParams.update(
    {
        "font.size": 12,
        "axes.titlesize": 14,
        "axes.labelsize": 13,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 11,
    }
)

fig, axes = plot_belief_vote_evolution(
    expected_mean=traj["expected_mean"],
    expected_precision=traj["expected_precision"],
    observations=traj["observations"],
    preference_params=traj["preference_params"],
    plurality_matrix=plur_win,
    quadratic_matrix=qv_win,
    candidate_labels=[f"C{i}" for i in range(n_cand)],
    shock_t=shock_t,
    title=f"Candidate win probability over time ({vt_config.num_voters} voters)",
    plurality_label="Plurality\nP(win)",
    quadratic_label="Quadratic\nP(win)",
    plurality_cbar="P(win)",
    quadratic_cbar="P(win)",
    vmax=vmax,
)
plt.show()